# NB04 — Phase 3b: the capacity ladder and the Oracle

**Plan v2 §7.2, §3.1, and the exactness check from §6. Budget: 2.0 h. Needs the same
single-GPU 80 GB pod as NB03.**

**What the ladder is measuring changed with the code audit.** `C0` is not the bottom rung of
a ladder we invented — it is the paper's own patching configuration, which builds no
projector on either side (Appendix F.1). And `C128` is the rank its projectors are actually
capped at, because `model/utils.py:252–253` makes every trainable projector a LoRA target
module (F.3). So two rungs of this ladder are the paper's real capacity and the rest are
ours. That is what makes the ladder necessary rather than ornamental: without it, "the
self-explainer advantage collapses under rotation" and "the paper's setup cannot represent
`Q^{-1}`" are indistinguishable claims.

`C128` also earns its row as a *bug detector*. It is exactly what `Cfull` degrades to if the
input map is ever routed through PEFT, so running it deliberately turns a silent failure into
a measured data point — and §0 below asserts, before any GPU time, that `Cfull` really is
full-rank.

Capacity is not a nuisance to be controlled away, it is the instrument. If the explainer's
ingestion path cannot represent `Q^{-1}`, a collapse under rotation says something about
parameter count and nothing about basis.

| Arm | Input map on `v` | Represents `Q^{-1}`? | Trainable |
|---|---|---|---|
| `C0` | none | No | — |
| `C8` / `C64` / `C512` | rank 8 / 64 / 512 | No / No / partially | yes |
| `Cfull` | full-rank `d×d`, init `I` | exactly | yes |
| `Oracle` | frozen at `Q^T` | exactly | **no** |

**`Oracle` does two jobs.** It separates *representability* from *learnability*: the gap
`Oracle − Cfull/R-Q` is the cost of having to learn the map rather than being handed it. And
it gives an exact bug check — `Oracle` under R-Q is algebraically identical to `C0` under
R-id, since `Q^T(Qv) = v` and the rest of the network is untouched.

**The honest reading is narrower than "the minimum rank that recovers measures how much of the
advantage is coordinate-frame."** At `N = 128` you are fitting a `4096²` map from ~128 injected
vectors; low ranks will win at low `N` and high ranks at high `N` for pure bias–variance
reasons, with the basis effect sitting inside that. The ladder *plus* `Oracle` is what
disentangles them. Say so in the write-up rather than letting a reviewer say it for you.


In [ ]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


In [ ]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# Same training path as NB03, walked up the capacity ladder.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=80)

import se_config as C


In [ ]:
import json
import subprocess
import time

import pandas as pd
import torch

import rotate as R
import se_common as S

# cheap check that the ladder is a real capacity ladder before spending GPU hours on it
print(subprocess.run([sys.executable, f"{REPO_DIR}/se/test_input_map.py"],
                     capture_output=True, text=True).stdout)


## 0. Assert full-rank trainability before spending anything

Revised §7.2's first checklist item, and §7.5's:

> *"Assert full-rank trainability before running anything: parameter count check per §3.1's trap
> box."* — *"Log trainable parameter counts per arm and confirm `Cfull` is not inside
> `target_modules`. A one-line check that protects the entire result."*

The failure this guards against is silent. `model/utils.py:252–253` of the paper's release appends
every trainable projector to LoRA's `target_modules`; under `lora_r: 128` a "full-rank" projector
is then frozen-at-init plus a rank-128 update, which at `d = 4096` cannot approximate `Q^T`. The
paper's footnote 7 says so and is accurate. If our `Cfull` ever lands in `target_modules`, the
central arm silently becomes `C128`, every curve still looks plausible, and the experiment
measures LoRA rank instead of basis.

So: `se_common.run_training` asserts the input-map parameter count on **every** run and records it
in `metrics.json`; `se/test_input_map.py` proves the assertion can fail (both the rank-capped and
the LoRA-wrapped shapes); and the cell below states the arithmetic at 8B scale before the ladder
starts.


In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(C.TARGET_MODEL_ID)
d = cfg.hidden_size

rows = []
for cap in C.CAPACITIES:
    n_params = S.expected_input_map_params(cap, d, d)
    rows.append({"arm": cap, "rank": C.CAPACITY_RANK[cap],
                 "trainable_params": n_params,
                 "params_M": round(n_params / 1e6, 2),
                 "pct_of_Cfull": round(100 * n_params
                                       / S.expected_input_map_params("Cfull", d, d), 2),
                 "status": C.CAPACITY_STATUS[cap]})
ladder_budget = pd.DataFrame(rows)
print(ladder_budget.to_string(index=False))

cfull = S.expected_input_map_params("Cfull", d, d)
c128 = S.expected_input_map_params("C128", d, d)

checks = [
    ("input map is outside LoRA target_modules",
     "input_map" not in C.LORA_TARGET_MODULES),
    (f"Cfull trains {C.N_LAYER_CHUNKS} x {d}^2 = {cfull:,} parameters",
     cfull == C.N_LAYER_CHUNKS * d * d),
    (f"C128 is a real cap: {c128:,} is {100 * c128 / cfull:.1f}% of Cfull",
     c128 < cfull / 10),
    ("the runtime assertion exists and is wired into run_training",
     hasattr(S, "check_input_map_trainability")),
]
print()
for name, ok in checks:
    print(f"  [{'ok  ' if ok else 'FAIL'}] {name}")
    assert ok, name

print(f"\nThe paper's projectors are per LAYER, so §3.1 quotes {32 * d * d / 1e6:.0f}M for 32 of "
      f"them.\nOurs are per layer CHUNK (N_LAYER_CHUNKS = {C.N_LAYER_CHUNKS}), so the same "
      f"argument lands at {cfull / 1e6:.0f}M.\nThe shape of the check is what matters: a few "
      f"million where tens of millions belong means a rank cap.")


## 1. The exactness check: `Oracle`/R-Q ≡ `C0`/R-id

v2 §6: *"They are algebraically the same computation. If the metrics differ beyond bf16 noise,
stop and debug — everything downstream is contaminated."*

`Oracle` is frozen at `Q^T` and receives `Qv`; `C0` has no map and receives `v`. Both hand the
model exactly `v`. With matched seeds and data order the two runs are the same computation
routed differently, so any disagreement is plumbing.

This is the cheapest bug check in the project and it runs before the ladder, not after.


In [ ]:
tokenizer = S.load_tokenizer()
Q, _ = R.load_rotation(f"{C.ROTATION_DIR}/Q_seed{C.Q_SEED}.pt")
datasets = {arm: S.load_ready_dataset(arm) for arm in ("identity", "Q")}

N_CHECK = min(C.N_TRAIN_LADDER)
oracle_mats = S.oracle_matrices(Q)

print(f"=== Oracle · R-Q · n={N_CHECK} " + "=" * 30)
kw_o = dict(tokenizer=tokenizer, dataset=datasets["Q"], ridge_matrices=oracle_mats)
S.run_training(N_CHECK, "Q", "Oracle", "oracle", **kw_o)
oracle_scores = S.eval_run(N_CHECK, "Q", "Oracle", "oracle", **kw_o)

print(f"\n=== C0 · R-id · n={N_CHECK} " + "=" * 30)
kw_c = dict(tokenizer=tokenizer, dataset=datasets["identity"])
S.run_training(N_CHECK, "identity", "C0", "identity", **kw_c)
c0_scores = S.eval_run(N_CHECK, "identity", "C0", "identity", **kw_c)


In [ ]:
METRICS = ["exact_match", "has_changed_f1", "content_match"]
deltas = {m: oracle_scores[m] - c0_scores[m] for m in METRICS}

print("EXACTNESS CHECK — Oracle/R-Q vs C0/R-id")
print("=" * 60)
for m in METRICS:
    print(f"  {m:>16}: Oracle {oracle_scores[m]:.4f}  C0 {c0_scores[m]:.4f}  "
          f"delta {deltas[m]:+.4f}")

TOL = 0.02      # bf16 noise plus one eval set's worth of decode nondeterminism
passed = all(abs(v) <= TOL for v in deltas.values())
with open(f"{C.REPORTS_DIR}/exactness_check.json", "w") as f:
    json.dump({"oracle": oracle_scores, "c0": c0_scores, "deltas": deltas,
               "tolerance": TOL, "passed": passed}, f, indent=2)

if passed:
    print(f"\nPASS — the two routes agree within {TOL}. The injection path is wired correctly.")
else:
    print(f"\nFAIL — these are algebraically the same computation. Stop and debug; "
          f"everything downstream is contaminated.")
    print("Suspects: chunk_id misalignment, a dtype cast before the map rather than after,")
    print("or the rotated dataset not matching the Q loaded here.")
assert passed, "exactness check failed"


## 2. What each rung costs

The rung costs, against the LoRA budget they sit beside.

v2 §3 describes the full-rank map as `d × d`; the pipeline learns one **per layer chunk**
(`N_LAYER_CHUNKS = 4`), so `Cfull` is ~4× a single matrix and exceeds the LoRA budget rather
than rounding to nothing. That does not weaken the case for `Cfull` — it is exactly the
parameter class in which `Q^T` lives — but "the map is negligible" is not an available
argument. The ladder is.


In [ ]:
# cfg and d come from §0 above, which loaded them for the trap-box assertion
head_dim = getattr(cfg, "head_dim", d // cfg.num_attention_heads)
shapes = {"q_proj": (d, cfg.num_attention_heads * head_dim),
          "k_proj": (d, cfg.num_key_value_heads * head_dim),
          "v_proj": (d, cfg.num_key_value_heads * head_dim),
          "o_proj": (cfg.num_attention_heads * head_dim, d),
          "gate_proj": (d, cfg.intermediate_size), "up_proj": (d, cfg.intermediate_size),
          "down_proj": (cfg.intermediate_size, d)}
lora_params = cfg.num_hidden_layers * sum(
    C.LORA_R * (i + o) for k, (i, o) in shapes.items() if k in C.LORA_TARGET_MODULES)

rows = []
for cap in C.CAPACITIES:
    total = S.expected_input_map_params(cap, d, d)
    rows.append({"arm": cap, "rank": C.CAPACITY_RANK[cap],
                 "trainable": cap != "Oracle",
                 "map_params_M": total / 1e6,
                 "pct_of_lora": 100 * total / lora_params,
                 "status": C.CAPACITY_STATUS[cap]})
print(f"LoRA budget: {lora_params/1e6:.1f}M trainable params")
print(f"The paper's own LoRA rank is {C.PAPER_LORA_R}; ours is {C.LORA_R}, inherited from the")
print("base replication. The rank that matters for the ladder is the one on the input MAP.")
pd.DataFrame(rows).round(2)


## 3. Run the ladder

Rotated activations at two `N`. The identity arm is run at `C0` and `Cfull` only — the
intermediate rungs are uninformative under R-id, where no inversion is needed.

The rungs are `C0`, `C8`, `C128`, `C512`, `Cfull` (revised §7.2). `C128` replaced `C64`
because 128 is the rank the paper's own projectors are capped at, which makes it the right
comparison for any claim about the paper's numbers; `C64` was an arbitrary point on a log
grid and bought nothing that `C8` and `C512` do not bracket.

**If this runs long**, v2 §8's cut order applies within the notebook: drop `C512` first, then
`C8`, keeping `C0`, `C128`, `Cfull` and `Oracle`. Those four still distinguish "the paper's
configuration", "the paper's rank cap", "representable but not learnable", and "learned
it", which is the shape of the finding.


In [ ]:
LADDER = ["C0", "C8", "C128", "C512", "Cfull"]

jobs = [("Q", cap, C.CAPACITY_DEFAULT_INIT[cap], None) for cap in LADDER]
jobs += [("Q", "Oracle", "oracle", oracle_mats)]
jobs += [("identity", cap, C.CAPACITY_DEFAULT_INIT[cap], None) for cap in ("C0", "Cfull")]

ladder_results = []
for n in C.N_TRAIN_LADDER:
    for rot, cap, init, mats in jobs:
        t0 = time.time()
        print(f"\n=== {C.ROTATION_LABEL[rot]} · {cap} · n={n} " + "=" * 26)
        kw = dict(tokenizer=tokenizer, dataset=datasets[rot], ridge_matrices=mats)
        S.run_training(n, rot, cap, init, **kw)
        scores = S.eval_run(n, rot, cap, init, **kw)
        ladder_results.append(scores)
        print(f"  exact_match {scores['exact_match']:.3f} "
              f"| content_match {scores['content_match']:.3f} [{(time.time()-t0)/60:.1f} min]")

ladder = pd.DataFrame(ladder_results)
ladder.to_csv(f"{C.REPORTS_DIR}/capacity_ladder.csv", index=False)

# §7.5's first item, verified from what the runs actually recorded rather than from intent
audits = []
for rot, cap, init, _ in jobs:
    for n in C.N_TRAIN_LADDER:
        path = f"{C.run_dir('patching', rot, cap, init, n)}/metrics.json"
        if not os.path.exists(path):
            continue
        m = json.load(open(path))
        audits.append({"rotation": rot, "capacity": cap, "n_train": n,
                       "trainable": m.get("input_map_trainable_params"),
                       "expected": m.get("input_map_expected_params"),
                       "lora_wrapped": bool(m.get("input_map_lora_wrapped")),
                       "ok": m.get("full_rank_ok")})
if audits:
    ad = pd.DataFrame(audits)
    print("\ninput-map audit, per run (§7.5):")
    print(ad.to_string(index=False))
    assert ad.ok.fillna(False).all(), (
        "a run's input map was rank-capped or LoRA-wrapped — the arm is not what it says "
        "it is (Appendix F.3). Stop and fix target_modules.")

ladder[["rotation", "capacity", "n_train", "exact_match", "content_match"]]


## 4. The recovery plot

Three horizontal references: `Cfull`/R-id (what the unrotated explainer achieves), `Oracle`
(what a *perfect* inverse achieves without having to learn it), and the **no-activation
floor** from NB03 §1 (what the explainer scores with no information in `v` at all). The
distance between the rotated ladder and `Oracle` is learnability; between `Oracle` and
`Cfull`/R-id is whatever the rotation costs that inverting the frame does not fix; and the
distance from the floor is the only part of the axis where anything is being measured.

`C0` and `C128` are marked on the rank axis as the paper's configuration and the paper's rank
cap — revised §7.2 calls this "the single most legible way to show why the ladder is
necessary rather than ornamental."


In [ ]:
import matplotlib.pyplot as plt

RANK_X = {"C0": 0.5, "C8": 8, "C128": 128, "C512": 512, "Cfull": 4096}
PAPER_RUNGS = {"C0", "C128"}          # F.1: no projector at all; F.3: rank-128 cap
prereg = json.load(open(f"{C.REPORTS_DIR}/preregistration.json"))
metric = prereg["primary_metric"]

# the floor NB03 measured, which is where this axis actually starts
floor_path = f"{C.REPORTS_DIR}/no_activation_floor.csv"
floors = {}
if os.path.exists(floor_path):
    fdf = pd.read_csv(floor_path)
    floors = dict(zip(fdf.n_train.astype(int), fdf[metric]))
else:
    print("no no_activation_floor.csv — run NB03 §1 first; plotting without the floor")

fig, axes = plt.subplots(1, len(C.N_TRAIN_LADDER),
                         figsize=(6.4 * len(C.N_TRAIN_LADDER), 4.8), squeeze=False)
for ax, n in zip(axes[0], C.N_TRAIN_LADDER):
    sub = ladder[(ladder.rotation == "Q") & (ladder.n_train == n)
                 & (ladder.capacity.isin(RANK_X))]
    sub = sub.assign(x=sub.capacity.map(RANK_X)).sort_values("x")
    ax.plot(sub.x, sub[metric], marker="o", color="#c0392b", label="rotated (R-Q)")

    for cap, rot, color, style, name in [
        ("Cfull", "identity", "#1b6ca8", "--", "Cfull · R-id"),
        ("Oracle", "Q", "#27ae60", "-.", "Oracle (frozen Q^T)"),
    ]:
        ref = ladder[(ladder.rotation == rot) & (ladder.capacity == cap)
                     & (ladder.n_train == n)][metric]
        if len(ref):
            ax.axhline(ref.iloc[0], color=color, linestyle=style, label=name)
    if n in floors:
        ax.axhline(floors[n], color="black", linestyle=":", linewidth=1.4,
                   label="no-activation floor")

    # the paper's own rungs, called out on the axis
    for cap in PAPER_RUNGS:
        if cap in RANK_X:
            ax.axvline(RANK_X[cap], color="#8e44ad", alpha=0.25, linewidth=6)
    ax.set_xscale("log", base=2)
    ax.set_xticks(list(RANK_X.values()))
    ax.set_xticklabels([f"{k}\n(paper)" if k in PAPER_RUNGS else k for k in RANK_X])
    ax.set_xlabel("rank of trainable input map on v")
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_title(f"N_TRAIN = {n}")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=8)
fig.suptitle("Capacity recovery under rotation. Shaded rungs are the paper's own configuration "
             "(C0: no projector,\nF.1) and its rank cap (C128, F.3). Caption must state the "
             "bias-variance caveat: at low N, low ranks win\nfor reasons unrelated to basis — "
             "Oracle is what separates representability from learnability.")
fig.tight_layout()
fig.savefig(f"{C.FIGURES_DIR}/capacity_recovery.png", dpi=150)
plt.show()


In [ ]:
frac = prereg["thresholds"]["recovery_fraction"]


def get(rot, cap, n):
    r = ladder[(ladder.rotation == rot) & (ladder.capacity == cap) & (ladder.n_train == n)]
    return float(r.iloc[0][metric]) if len(r) else None


ladder_rows = []
for n in C.N_TRAIN_LADDER:
    ref = get("identity", "Cfull", n)
    oracle = get("Q", "Oracle", n)
    fl = floors.get(n)
    if ref is None:
        continue

    def retained(v):
        return S.fraction_retained(v, fl, ref) if (fl is not None and v is not None) else None

    print(f"N = {n}   (floor {fl if fl is None else round(fl, 3)}, "
          f"reference {ref:.3f} — the axis is {'' if fl is None else round(ref - fl, 3)} wide)")
    for cap in LADDER + ["Oracle"]:
        v = get("Q", cap, n)
        if v is None:
            continue
        r = retained(v)
        tag = "  <- the paper's configuration" if cap == "C0" else (
            "  <- the paper's rank cap" if cap == "C128" else "")
        print(f"  R-Q {cap:>7}: {metric} {v:.3f}"
              + (f"   retained {r:+.2f}" if r is not None else "") + tag)
        ladder_rows.append({"n_train": n, "capacity": cap, "raw": v, "retained": r,
                            "floor": fl, "reference": ref})

    recovered = [cap for cap in LADDER if (retained(get("Q", cap, n)) or -9) >= frac]
    print(f"  recovery threshold   : {recovered[0] if recovered else 'none recovered'} "
          f"(>= {frac:.0%} of the activation's contribution)")
    if oracle is not None:
        cfull_q = get("Q", "Cfull", n)
        if cfull_q is not None and fl is not None:
            print(f"  learnability cost    : Oracle - Cfull/R-Q = "
                  f"{(retained(oracle) - retained(cfull_q)):+.2f} of the contribution "
                  f"({oracle - cfull_q:+.3f} raw)")
        print(f"  irreducible rotation cost: Cfull/R-id - Oracle = {ref - oracle:+.3f} raw")
    print()

if ladder_rows:
    pd.DataFrame(ladder_rows).to_csv(f"{C.REPORTS_DIR}/capacity_ladder_normalized.csv",
                                     index=False)

print("Read this against §3.5's caveat, and put it in the figure caption: at N = 512 a 4096^2 map")
print("is being fit from 512 injected vectors, so low ranks win at low N for bias-variance")
print("reasons with the basis effect sitting inside that. Oracle plus the ladder is what")
print("disentangles them — the recovery threshold alone does not.")


Next: **NB05** prices the alternative recipe, including the arm that needs no per-target
training at all.
